# SocraticLM и TutorRL в Yandex DataSphere: скачать и запустить

Ноутбук скачивает в постоянное хранилище проекта две модели-репетитора и запускает их на GPU:
- **SocraticLM** — `Qwen2.5-Math-7B-Instruct`, дообученная на датасете SocraTeach (сократический диалог по математике).
- **TutorRL-7B** — `Qwen2.5-7B-Instruct`, обученная через multi-turn RL (GRPO) в фреймворке PedagogicalRL (репетиторство с симуляцией ученика).

**Порядок работы**
1. Шаги 0–6 удобно выполнять на конфигурации **c1.4**. Для скачивания видеокарта не нужна, а час c1.4 стоит в 18 раз дешевле g1.1. Можно и сразу на g1.1, но если скачивание затянется, вы будете платить за простаивающую V100.
2. Затем переключитесь на **g1.1** (V100, 32 ГБ) и выполните шаги 7–15. Обе модели суммарно занимают около 30 ГБ, поэтому на V100 их лучше загружать по очереди, освобождая память. На **g2.1** (A100, 80 ГБ) можно держать обе одновременно.

**Перед запуском нужно место.** Каждая модель занимает ~15,2 ГБ, то есть под две нужно около 31 ГБ.

Добавлен отдельный шаг скачивания третьей модели TR-finetuned-adversary перед переключением на GPU.


## Шаг 0. Настройки

Единственное место, где задаются пути. После переключения ВМ эту ячейку нужно выполнить заново.

In [ ]:
# Вариант А: хранилище проекта (после увеличения до 50 ГБ)
MODELS_ROOT = "/home/jupyter/datasphere/project/models"
# Вариант Б: файловое хранилище. Вставьте путь из Copy path (без /models на конце)
# MODELS_ROOT = "<путь_к_файловому_хранилищу>"

# --- SocraticLM ---
SOCRATICLM_DIR = MODELS_ROOT + "/SocraticLM"
# Открытая копия: файлы побайтно совпадают с официальным CogBase-USTC/SocraticLM, токен не нужен
SOCRATICLM_REPO_ID = "CogBase-USTC/Qwen2.5-Math-7B-Instruct-SocraticLM"
# Официальный репозиторий с закрытым доступом (нужен секрет HFTOKEN, см. шаг 3)
# SOCRATICLM_REPO_ID = "CogBase-USTC/SocraticLM"

# --- TutorRL ---
TUTORRL_DIR = MODELS_ROOT + "/TutorRL-7B"
TUTORRL_REPO_ID = "eth-nlped/TutorRL-7B"  # публичный, токен не нужен

# Куда клонировать репозиторий фреймворка (нужен только для обучения/оценки)
PEDAGOGICALRL_DIR = "/home/jupyter/datasphere/project/PedagogicalRL"

## Шаг 1. Окружение и свободное место

Для сетевых дисков цифра свободного места бывает приблизительной. Точный размер хранилища показан на странице проекта. Под две модели нужно около 31 ГБ.

In [ ]:
import os, sys, shutil

print("Python", sys.version.split()[0])
try:
    import torch
    print("torch", torch.__version__, "| CUDA доступна:", torch.cuda.is_available())
except ImportError:
    print("torch не найден")

for p in [MODELS_ROOT, "/home/jupyter/datasphere/project", "/home/jupyter/datasphere"]:
    if os.path.isdir(p):
        free = shutil.disk_usage(p).free / 1e9
        print(f"\n{p}: свободно примерно {free:.0f} ГБ")
        print("   содержимое:", sorted(os.listdir(p))[:30])

## Шаг 2. Библиотеки

В стандартном образе DataSphere стоит torch 2.0.1, а свежие версии transformers требуют torch не ниже 2.5. Поэтому ставим transformers 4.46.2: на ней авторы SocraticLM обучали модель, и она работает со старым torch. accelerate закреплён на версии того же времени, так что torch не обновляется. TutorRL-7B тоже совместима с transformers 4.46.x (это Qwen2.5).

После переключения на GPU выполните эту ячейку ещё раз. Если пакеты уже на месте, она отработает за несколько секунд.

In [ ]:
%pip install -q "transformers==4.46.2" "accelerate==1.1.1"

import transformers, accelerate, huggingface_hub, torch
print("transformers", transformers.__version__, "| accelerate", accelerate.__version__,
      "| huggingface_hub", huggingface_hub.__version__, "| torch", torch.__version__)

## Шаг 3. Скачивание весов SocraticLM

По умолчанию качается открытая копия, вход на Hugging Face не нужен. Если нужен именно официальный `CogBase-USTC/SocraticLM`:
1. На huggingface.co откройте страницу модели и примите условия доступа.
2. Settings → Access Tokens: создайте токен с правом Read.
3. В DataSphere: «Ресурсы проекта» → «Секрет» → «Создать», имя `HFTOKEN`, в поле «Значение» вставьте токен. Секреты видны в ноутбуке как переменные окружения.
4. В шаге 0 раскомментируйте вторую строку с `SOCRATICLM_REPO_ID`.

Уже скачанные файлы повторно не качаются: если ячейка упала, просто запустите её снова.

In [ ]:
import os, shutil
from huggingface_hub import snapshot_download

os.makedirs(SOCRATICLM_DIR, exist_ok=True)
free = shutil.disk_usage(SOCRATICLM_DIR).free / 1e9
if free < 16:
    print(f"Внимание: свободно примерно {free:.0f} ГБ, а нужно около 16 под одну модель. Увеличьте хранилище.")

path_socratic = snapshot_download(
    repo_id=SOCRATICLM_REPO_ID,
    local_dir=SOCRATICLM_DIR,
    token=os.environ.get("HFTOKEN"),  # для открытой копии остаётся None
    max_workers=4,
)
print("SocraticLM скачан в", path_socratic)

## Шаг 4. Проверка файлов SocraticLM

In [ ]:
import json, os

files = sorted(f for f in os.listdir(SOCRATICLM_DIR) if not f.startswith("."))
total = 0
for f in files:
    size = os.path.getsize(os.path.join(SOCRATICLM_DIR, f))
    total += size
    print(f"{size / 1e9:7.2f} ГБ  {f}")
print(f"Итого {total / 1e9:.1f} ГБ, ожидается около 15,2")

index = json.load(open(os.path.join(SOCRATICLM_DIR, "model.safetensors.index.json")))
shards = sorted(set(index["weight_map"].values()))
missing = [s for s in shards if not os.path.exists(os.path.join(SOCRATICLM_DIR, s))]
print("Частей весов:", len(shards), "| не хватает:", missing or "ничего")

## Шаг 5. Скачивание весов TutorRL-7B

Репозиторий `eth-nlped/TutorRL-7B` публичный, токен не требуется. Модель весит около 15 ГБ. Логика та же: если ячейка упала — просто перезапустите, докачает с места остановки.

In [ ]:
import os, shutil
from huggingface_hub import snapshot_download

os.makedirs(TUTORRL_DIR, exist_ok=True)
free = shutil.disk_usage(TUTORRL_DIR).free / 1e9
if free < 16:
    print(f"Внимание: свободно примерно {free:.0f} ГБ, а нужно около 16 под одну модель. Увеличьте хранилище.")

path_tutorrl = snapshot_download(
    repo_id=TUTORRL_REPO_ID,
    local_dir=TUTORRL_DIR,
    token=os.environ.get("HFTOKEN"),  # не обязателен для публичной модели
    max_workers=4,
)
print("TutorRL-7B скачан в", path_tutorrl)

## Шаг 6. Проверка файлов TutorRL-7B

In [ ]:
import json, os

files_t = sorted(f for f in os.listdir(TUTORRL_DIR) if not f.startswith("."))
total_t = 0
for f in files_t:
    size = os.path.getsize(os.path.join(TUTORRL_DIR, f))
    total_t += size
    print(f"{size / 1e9:7.2f} ГБ  {f}")
print(f"Итого {total_t / 1e9:.1f} ГБ, ожидается около 15")

index_t = json.load(open(os.path.join(TUTORRL_DIR, "model.safetensors.index.json")))
shards_t = sorted(set(index_t["weight_map"].values()))
missing_t = [s for s in shards_t if not os.path.exists(os.path.join(TUTORRL_DIR, s))]
print("Частей весов:", len(shards_t), "| не хватает:", missing_t or "ничего")

## Третья модель: TR-finetuned-adversary

Выполните эти ячейки после шагов 0 и 2, до загрузки моделей на GPU.
Уже скачанные репетиторы повторно скачивать не нужно.
Репозиторий: `ml4ed-epfl/TR-finetuned-adversary`.

Код получает размер файлов и фиксирует SHA ревизии до скачивания. Если опубликован
LoRA-адаптер, скачивает также указанную в его конфигурации базовую модель.
Доступность, архитектура и формат промпта не проверены в текущей среде: Hugging Face
вернул 403. Не предполагаем заранее, что это 7B-модель или публичный репозиторий.

In [ ]:
%pip install -q "huggingface_hub>=0.26,<1.0" "peft==0.13.2" sentencepiece

In [ ]:
import os, json, shutil
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download, snapshot_download

ADVERSARY_REPO_ID = "ml4ed-epfl/TR-finetuned-adversary"
ADVERSARY_DIR = Path(MODELS_ROOT) / "TR-finetuned-adversary"
# Повторный запуск использует ранее зафиксированную ревизию.
# Для намеренного обновления удалите только этот manifest после сохранения старой версии.
MANIFEST_PATH = Path(MODELS_ROOT) / "adversary_manifest.json"
hf_token = os.environ.get("HFTOKEN") or os.environ.get("HF_TOKEN")
api = HfApi(token=hf_token)
patterns = ["*.json", "*.safetensors", "*.model", "*.txt", "*.tiktoken", "*.jinja", "README.md"]


def repo_info(repo_id, revision=None):
    return api.model_info(repo_id, revision=revision, files_metadata=True)


def selected_files(info):
    import fnmatch
    return [f for f in info.siblings if any(fnmatch.fnmatch(f.rfilename, p) for p in patterns)]


def download_checked(info, destination):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    files = selected_files(info)
    if not files or any(f.size is None for f in files):
        raise RuntimeError("Не удалось определить размеры файлов модели")
    missing_bytes = sum(f.size for f in files
                        if not (destination / f.rfilename).is_file()
                        or (destination / f.rfilename).stat().st_size != f.size)
    free = shutil.disk_usage(destination).free
    print(f"{info.id}: скачать до {missing_bytes/1e9:.2f} GB, свободно {free/1e9:.2f} GB")
    if free < missing_bytes + 2_000_000_000:
        raise RuntimeError("Недостаточно места для модели и запаса 2 GB")
    snapshot_download(repo_id=info.id, revision=info.sha, local_dir=str(destination),
                      token=hf_token, allow_patterns=patterns, max_workers=4)
    invalid = [f.rfilename for f in files if not (destination / f.rfilename).is_file()
               or (destination / f.rfilename).stat().st_size != f.size]
    if invalid:
        raise RuntimeError(f"Отсутствуют или неполны файлы: {invalid}")
    return str(destination.resolve())

saved = json.loads(MANIFEST_PATH.read_text()) if MANIFEST_PATH.exists() else None
if saved and saved["repo_id"] != ADVERSARY_REPO_ID:
    raise ValueError("Manifest относится к другой модели")
info = repo_info(ADVERSARY_REPO_ID, saved["revision"] if saved else None)
files = {f.rfilename for f in info.siblings}
is_adapter = "adapter_config.json" in files
if not any(f.endswith(".safetensors") for f in files):
    raise RuntimeError("В репозитории нет safetensors; формат нужно проверить отдельно")
manifest = {"repo_id": ADVERSARY_REPO_ID, "revision": info.sha,
            "kind": "adapter" if is_adapter else "full",
            "model_dir": str(ADVERSARY_DIR.resolve()), "base_dir": None}
if is_adapter:
    config_path = hf_hub_download(ADVERSARY_REPO_ID, "adapter_config.json",
                                  revision=info.sha, token=hf_token)
    adapter_config = json.loads(Path(config_path).read_text())
    base_id = adapter_config.get("base_model_name_or_path")
    if not base_id or base_id.startswith(("/", ".")):
        raise RuntimeError("У адаптера нет доступного HF ID базовой модели")
    base_revision = saved.get("base_revision") if saved else adapter_config.get("revision")
    base_info = repo_info(base_id, base_revision)
    base_dir = Path(MODELS_ROOT) / "TR-adversary-base"
    manifest.update(base_repo_id=base_id, base_revision=base_info.sha,
                    base_dir=download_checked(base_info, base_dir))
manifest["model_dir"] = download_checked(info, ADVERSARY_DIR)
MANIFEST_PATH.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("Готово. Manifest:", MANIFEST_PATH)

### Проверка конфигурации без загрузки весов на GPU

Если архитектура не поддерживается `transformers==4.46.2`, эта ячейка остановится.
Тогда нужна совместимая версия transformers в отдельном окружении. Не включайте
`trust_remote_code=True` автоматически. Для gated-модели примите условия на Hugging Face
и задайте `HFTOKEN` в секретах DataSphere, без вставки токена в код.

In [ ]:
from transformers import AutoConfig, AutoTokenizer

manifest = json.loads(MANIFEST_PATH.read_text())
config_dir = manifest["base_dir"] or manifest["model_dir"]
cfg = AutoConfig.from_pretrained(config_dir, local_files_only=True, trust_remote_code=False)
# У адаптера tokenizer может находиться только в базовой модели.
tokenizer_dir = manifest["model_dir"]
if not (Path(tokenizer_dir) / "tokenizer_config.json").exists():
    tokenizer_dir = config_dir
adversary_tokenizer = AutoTokenizer.from_pretrained(
    tokenizer_dir, local_files_only=True, trust_remote_code=False)
if not adversary_tokenizer.chat_template:
    raise RuntimeError("Нет chat_template: нужен документированный формат промпта модели")
print("Архитектура:", cfg.model_type, cfg.architectures)
print("Контекст:", getattr(cfg, "max_position_embeddings", None))
print("Формат:", manifest["kind"], "| ревизия:", manifest["revision"])
# Это проверка конфигурации, а не подтверждение качества атак.

## Переключение на GPU

1. Остановите текущую ВМ: страница проекта, виджет «Запущенные операции», кнопка «Остановить JupyterLab и ВМ».
2. Снова откройте проект и при первом запуске ячейки выберите **g1.1** (V100, 32 ГБ видеопамяти), она доступна без пополнения баланса. **g2.1** (A100, 80 ГБ) открывается после пополнения баланса от 500 ₽.
3. Состояние ядра при смене ВМ не сохраняется. Выполните заново шаги 0 и 2, затем шаги 7–15.

На V100 суммарно 32 ГБ, а две модели занимают ~30 ГБ — поэтому ниже загрузка сделана так, что модели грузятся **по очереди**: перед загрузкой второй вызывается `free_models()`, чтобы освободить память. На A100 обе помещаются одновременно.

## Шаг 7. Общие хелперы для загрузки и выгрузки моделей

`pick_dtype()` выбирает float16 для V100/T4 (compute capability < 8, там нет bfloat16) и bfloat16 для A100 и новее.

`load_model(path)` возвращает пару (tokenizer, model) и печатает занятую видеопамять.

`free_models()` выгружает модели из глобальных переменных и очищает кэш CUDA. Вызывайте перед загрузкой второй модели на V100.

In [ ]:
import gc, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

assert torch.cuda.is_available(), "GPU не найден: выберите конфигурацию g1.1 или g2.1"

def pick_dtype():
    major, _ = torch.cuda.get_device_capability(0)
    return torch.bfloat16 if major >= 8 else torch.float16  # V100 (7.0) и T4 (7.5) без bfloat16

def load_model(path):
    tok = AutoTokenizer.from_pretrained(path)
    mdl = AutoModelForCausalLM.from_pretrained(
        path, torch_dtype=pick_dtype(), device_map={"": 0}
    )
    mdl.eval()
    print(f"Загружено: {path} | dtype={next(mdl.parameters()).dtype} | "
          f"видеопамяти занято: {torch.cuda.memory_allocated() / 1e9:.1f} ГБ")
    return tok, mdl

def free_models():
    for name in ["model_socratic", "model_tutorrl"]:
        if name in globals():
            del globals()[name]
    for name in ["tokenizer_socratic", "tokenizer_tutorrl"]:
        if name in globals():
            del globals()[name]
    gc.collect()
    torch.cuda.empty_cache()
    print(f"Память освобождена. Занято сейчас: {torch.cuda.memory_allocated() / 1e9:.1f} ГБ")

print(torch.cuda.get_device_name(0), "|", pick_dtype())

## Шаг 8. Загрузка SocraticLM на GPU

V100 не поддерживает bfloat16, поэтому на g1.1 модель загружается во float16. На A100 код сам выберет bfloat16. Чтение 15 ГБ с сетевого диска может занять несколько минут.

In [ ]:
tokenizer_socratic, model_socratic = load_model(SOCRATICLM_DIR)

## Шаг 9. Функция диалога с SocraticLM

`chat_socratic()` принимает историю сообщений в формате `[{"role": ..., "content": ...}]` и возвращает ответ модели. Генерация жадная, как в примере авторов (там `temperature=0`). Контекст модели 4096 токенов, промпт вместе с ответом должен в него укладываться.

In [ ]:
def chat_socratic(messages, max_new_tokens=1024):
    prompt = tokenizer_socratic.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    enc = tokenizer_socratic(prompt, return_tensors="pt").to(model_socratic.device)
    with torch.no_grad():
        out = model_socratic.generate(
            input_ids=enc["input_ids"],
            attention_mask=enc["attention_mask"],
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,  # в config.json модели стоит use_cache=false, кэш включаем явно
        )
    return tokenizer_socratic.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)

## Шаг 10. Проверка SocraticLM: решение задачи

Системный промпт и задача из карточки модели. Правильный ответ 72.

In [ ]:
messages = [
    {"role": "system", "content": "Please analyse and solve the following problem step by step."},
    {"role": "user", "content": "Natalia sold clips to 48 of her friends in April, and then she sold half "
                                "as many clips in May. How many clips did Natalia sell altogether in April and May?"},
]
print(chat_socratic(messages))

## Шаг 11. Проверка SocraticLM: сократический режим

Формат из карточки модели. В системном промпте роль учителя, в первом сообщении блоки `[Problem]`, `[Answer]` и `[Analysis]`. Реплики ученика добавляются в историю, и модель продолжает вести его вопросами.

In [ ]:
SOCRATIC_SYSTEM = ("You are a Socratic teacher, please guide me to solve the [Problem] "
                   "with heuristic questions based on the following information. \n")

problem = ("[Problem] Natalia sold clips to 48 of her friends in April, and then she sold half as many "
           "clips in May. How many clips did Natalia sell altogether in April and May? "
           "[Answer] 72 "
           "[Analysis] In May Natalia sold 48 / 2 = 24 clips. Altogether she sold 48 + 24 = 72 clips.")

dialog = [
    {"role": "system", "content": SOCRATIC_SYSTEM},
    {"role": "user", "content": problem},
]
reply = chat_socratic(dialog, max_new_tokens=512)
print("Учитель:", reply)

# Ответ ученика с типичной ошибкой и следующий ход учителя
dialog += [
    {"role": "assistant", "content": reply},
    {"role": "user", "content": "In May she sold 96 clips, because half as many means twice as many."},
]
print("\nУчитель:", chat_socratic(dialog, max_new_tokens=512))

## Шаг 12 (по желанию). SocraticLM на русском

Авторы заявляют поддержку английского и китайского. Поведение на русском лучше проверить сразу.

In [ ]:
ru_dialog = [
    {"role": "system", "content": SOCRATIC_SYSTEM},
    {"role": "user", "content": "[Problem] В классе 24 ученика, девочек на 6 больше, чем мальчиков. "
                                "Сколько в классе девочек? [Answer] 15 "
                                "[Analysis] Мальчиков (24 - 6) / 2 = 9, девочек 9 + 6 = 15."},
]
print(chat_socratic(ru_dialog, max_new_tokens=512))

## Шаг 13. Освобождение памяти и загрузка TutorRL-7B

На V100 32 ГБ две модели по 15 ГБ не помещаются вместе с активациями. Перед загрузкой TutorRL вызываем `free_models()`, который выгружает SocraticLM и очищает кэш CUDA. Если у вас A100 (g2.1) и хочется держать обе модели сразу — просто не вызывайте `free_models()` и сразу переходите к `load_model(TUTORRL_DIR)`, присвоив результат другим переменным.

In [ ]:
free_models()
tokenizer_tutorrl, model_tutorrl = load_model(TUTORRL_DIR)

## Шаг 14. Функция диалога с TutorRL-7B

`chat_tutorrl()` устроена так же, как `chat_socratic()`. TutorRL-7B обучали вести tutoring-диалог: она задаёт наводящие вопросы, а не сразу выдаёт ответ. Поддерживается стандартный chat-шаблон Qwen2.5 (`system` / `user` / `assistant`).

In [ ]:
def chat_tutorrl(messages, max_new_tokens=512):
    prompt = tokenizer_tutorrl.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    enc = tokenizer_tutorrl(prompt, return_tensors="pt").to(model_tutorrl.device)
    with torch.no_grad():
        out = model_tutorrl.generate(
            input_ids=enc["input_ids"],
            attention_mask=enc["attention_mask"],
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
        )
    return tokenizer_tutorrl.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)

## Шаг 15. Проверка TutorRL-7B

Простой запрос на помощь в решении уравнения. Ожидаемое поведение — репетитор задаёт встречный вопрос или предлагает первый шаг, а не сразу называет `x = 5`. Можно продолжить диалог, добавив ответ «ученика» и вызвав `chat_tutorrl` снова.

In [ ]:
tutor_dialog = [
    {"role": "user", "content": "Can you help me solve 3x + 5 = 20?"}
]
reply_t = chat_tutorrl(tutor_dialog)
print("Репетитор:", reply_t)

# Продолжение диалога: ответ ученика и следующий ход репетитора
tutor_dialog += [
    {"role": "assistant", "content": reply_t},
    {"role": "user", "content": "I subtracted 5 from both sides and got 3x = 15. What next?"},
]
print("\nРепетитор:", chat_tutorrl(tutor_dialog))

## Шаг 16 (по желанию). Фреймворк PedagogicalRL

Репозиторий [eth-lre/PedagogicalRL](https://github.com/eth-lre/PedagogicalRL) — это код обучения TutorRL через multi-turn RL (GRPO) и симуляцию ученика. Для простого инференса модели он не нужен. Клонируйте его, если планируете дообучение или воспроизведение оценки.

Запускайте на CPU-конфигурации (c1.4) или после того, как выгрузили модели — установка зависимостей подтянет свой torch и может конфликтовать с уже загруженным в ядро. Лучше делать это в отдельном окружении (создайте `.venv` внутри каталога репозитория).

In [ ]:
import os
if not os.path.isdir(PEDAGOGICALRL_DIR):
    !git clone https://github.com/eth-lre/PedagogicalRL.git {PEDAGOGICALRL_DIR}
else:
    print("Уже клонирован:", PEDAGOGICALRL_DIR)

!ls {PEDAGOGICALRL_DIR}

## Когда закончили

ВМ тарифицируется всё время, пока закреплена за ноутбуком, даже без вычислений. По умолчанию её отпускают только после 3 часов простоя, для g1.1 это около 1000 ₽. Остановите её вручную: страница проекта, «Запущенные операции», «Остановить JupyterLab и ВМ». Тайм-аут меняется в настройках проекта, параметр «Остановить простаивающую ВМ через».

## Если что-то пошло не так

- **Скачивание висит или падает с `ConnectTimeout`.** Если в настройках проекта задана подсеть, интернет работает только через NAT-шлюз. Если huggingface.co из DataSphere недоступен, скачайте веса на другой машине (`hf download <repo_id> --local-dir <dir>`) и перенесите через Object Storage и коннектор S3.
- **`401` или `403` на `CogBase-USTC/SocraticLM`.** Репозиторий закрытый: примите условия на странице модели и проверьте секрет `HFTOKEN`. Либо качайте открытую копию `Qwen2.5-Math-7B-Instruct-SocraticLM`.
- **`No space left on device`.** Не хватило хранилища. Увеличьте его и запустите шаги 3 и 5 снова, скачивание продолжится с места остановки.
- **Ответ состоит из сплошных «!!!!».** Обычно это признак переполнения во float16 на V100. Помогает A100 (g2.1) с bfloat16, либо уменьшение `max_new_tokens`.
- **`CUDA out of memory` при загрузке TutorRL.** Вы не вызвали `free_models()` перед загрузкой второй модели, либо параллельно остаются activations от SocraticLM. Выполните шаг 13 полностью.
- **Скачивание падает с ошибками `hf_xet`.** Перезапустите ядро, выполните `import os; os.environ["HF_HUB_DISABLE_XET"] = "1"` до импорта `huggingface_hub` и повторите шаги 0, 2, 3, 5.
- **Ошибки импорта после смены ВМ.** Повторите шаг 2.
- **Конфликт версий при `pip install -r requirements.txt` из PedagogicalRL.** Ставьте зависимости фреймворка в отдельный venv, а не в ядро DataSphere — иначе можно сломать рабочее окружение для инференса обеих моделей.

## О версиях моделей

**SocraticLM.** На Hugging Face лежит улучшенная реализация на Qwen2.5-Math-7B-Instruct. В статье (NeurIPS 2024) модель обучали на ChatGLM3-6B, а среди опубликованных весов авторов такой версии нет, поэтому числа из таблиц статьи на этой модели могут не совпасть. В коллекции авторов на Hugging Face есть ещё варианты на Llama-3.1-8B-Instruct и GLM-4-9B (демо). Датасет SocraTeach и скрипты оценки лежат в репозитории [Ljyustc/SocraticLM](https://github.com/Ljyustc/SocraticLM).

**TutorRL-7B.** Дообучение Qwen2.5-7B-Instruct через multi-turn GRPO по статье ETH NLPED. Репозиторий с кодом обучения и оценки — [eth-lre/PedagogicalRL](https://github.com/eth-lre/PedagogicalRL). Модель распространяется под лицензией Apache 2.0.